In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller  # 添加这行导入
import warnings

# 1. 设置工作目录和导入数据
data_path = "D:/Academic/RUC/Zheng/太空经济/CongestionTax/data/temp/satellite_data.csv"

# 使用pandas读取Excel文件
df = pd.read_csv(data_path)

# 确保数据正确加载
print("数据加载成功，前5行数据：")
print(df.head())

In [ ]:
# 删除2025年的数据
df = df[df['year'] != 2025]
print(f"删除2025年数据后的数据集大小: {df.shape}")
print("\n年份范围：")
print(df['year'].unique())

# 重新定义 tau
tau = 0.0000062

# 首先按国家和年份排序
df = df.sort_values(['country', 'year'])

# 按国家分组并获取上一年的X_total_cum
df['prev_total_launch'] = df.groupby('country')['X_total_cum'].shift(1)

# 计算LHS，使用prev_total_launch替代X_total_cum
df['LHS'] = 1 - tau * df['prev_total_launch'] - tau * (df['stock_launch'] + df['flow_launch'])

# 第一年的数据会是NaN，可以用当年的total_launch填充
df.loc[df['prev_total_launch'].isna(), 'LHS'] = 1 - tau * df.loc[df['prev_total_launch'].isna(), 'X_total_cum'] - tau * (df.loc[df['prev_total_launch'].isna(), 'stock_launch'] + df.loc[df['prev_total_launch'].isna(), 'flow_launch'])

In [ ]:
import pandas as pd
import statsmodels.api as sm

# 假设 df 是你的 DataFrame
df = df.sort_values(['country', 'year'])

# 创建存储估计结果的列
df['c_hat'] = float('nan')
df['gamma_hat'] = float('nan')
df['c_hat_constrained'] = float('nan')
df['gamma_hat_constrained'] = float('nan')

# 获取所有国家的列表
countries = df['country'].unique()

# 按国家逐步估计参数
for country in countries:
    # 获取当前国家的数据
    country_data = df[df['country'] == country]
    
    # 检查是否有足够的观测值（至少2个）
    if len(country_data[country_data['stock_launch'] != 0]) >= 2:
        # 进行回归
        X = sm.add_constant(country_data['stock_launch'])
        y = country_data['LHS']
        try:
            model = sm.OLS(y, X).fit()
            # 将结果存储回DataFrame
            df.loc[df['country'] == country, 'c_hat'] = model.params['const']
            df.loc[df['country'] == country, 'gamma_hat'] = model.params['stock_launch']
        except:
            continue

# 约束估计
for country in countries:
    country_data = df[df['country'] == country]
    
    # 检查是否有足够的观测值
    if country_data['c_hat'].notna().sum() > 0 and country_data['gamma_hat'].notna().sum() > 0:
        # OLS 估计 c_hat
        try:
            X = sm.add_constant(country_data['stock_launch'])
            y = country_data['c_hat']
            model = sm.OLS(y, X).fit()
            c_hat_constrained = model.params['const'] + model.params['stock_launch'] * country_data['stock_launch']
            df.loc[df['country'] == country, 'c_hat_constrained'] = c_hat_constrained.clip(lower=0)
        except:
            continue
        
        # OLS 估计 gamma_hat
        try:
            y = country_data['gamma_hat']
            model = sm.OLS(y, X).fit()
            gamma_hat_constrained = model.params['const'] + model.params['stock_launch'] * country_data['stock_launch']
            df.loc[df['country'] == country, 'gamma_hat_constrained'] = gamma_hat_constrained.clip(lower=0.01)
        except:
            continue

# 保存结果到 CSV 文件
df.to_csv('parameter_estimation_results.csv', index=False)
print("结果已保存到 'parameter_estimation_results.csv'")

In [18]:
# 3. 模拟分析
def simulation_analysis(df):
    # 创建新的预测结果列
    df['gamma_fixed'] = 0.00001  # 固定gamma值
    df['c_sim'] = df['LHS'] - (0.00001 * df['flow_launch'])
    df['LHS_pred'] = df['c_sim'] + (0.00001 * df['flow_launch'])
    
    # 创建发射过和未发射过的国家分组
    country_launches = df.groupby('country')['flow_launch'].sum()
    launched_countries = country_launches[country_launches > 0].index
    non_launched_countries = country_launches[country_launches == 0].index
    
    launched_data = df[df['country'].isin(launched_countries)]
    non_launched_data = df[df['country'].isin(non_launched_countries)]
    
    return df, launched_data, non_launched_data


In [19]:
def find_best_arima_order(series, max_p=3, max_d=2, max_q=3):
    """
    为时间序列找到最优的ARIMA模型阶数
    """
    # ADF检验
    adf_result = adfuller(series)
    d = 1 if adf_result[1] > 0.05 else 0
    
    # 尝试不同的p,d,q组合
    best_aic = float('inf')
    best_order_aic = None
    
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                model = ARIMA(series, order=(p, d, q))
                results = model.fit()
                if results.aic < best_aic:
                    best_aic = results.aic
                    best_order_aic = (p, d, q)
            except:
                continue
    
    return best_order_aic

def forecast_analysis(launched_data):
    """
    修改后的预测分析函数
    """
    current_year = launched_data['year'].max()
    years_to_2050 = 2050 - current_year
    predictions = pd.DataFrame()
    arima_orders = {}  # 用于存储每个国家的ARIMA阶数
    
    for country in launched_data['country'].unique():
        country_data = launched_data[launched_data['country'] == country]
        
        try:
            c_series = country_data['c_sim']
            
            
            if len(c_series) > 5:
                order = find_best_arima_order(c_series)
            else:
                order = (1,0,0)
            
            arima_orders[country] = order  # 记录ARIMA阶数
            
            model = ARIMA(c_series, order=order)
            results = model.fit()
            c_forecast = results.forecast(steps=years_to_2050)
            c_forecast = np.maximum(c_forecast, 1e-10)
            
            country_predictions = pd.DataFrame({
                'year': range(current_year + 1, 2051),
                'country': [country] * years_to_2050,
                'c_pred': c_forecast,
                'arima_order': [str(order)] * years_to_2050
            })
            
            
            predictions = pd.concat([predictions, country_predictions])
            
        except Exception as e:
            print(f"预测{country}时出错: {str(e)}")
            continue
    
    # 在最后统一输出每个国家使用的ARIMA阶数
    print("\n各国使用的ARIMA模型阶数:")
    for country, order in arima_orders.items():
        print(f"{country}: ARIMA{order}")
    
    return predictions

In [ ]:
# 5. 可视化函数
def plot_results(df, predictions):
    # 创建画布
    plt.figure(figsize=(15, 15))
    
    # c值预测趋势
    plt.subplot(3, 1, 1)
    for country in predictions['country'].unique():
        country_pred = predictions[predictions['country'] == country]
        plt.plot(country_pred['year'], country_pred['c_pred'], alpha=0.5, label=country)
    plt.title('Predicted c-values Trend (2023-2050)')
    plt.xlabel('Year')
    plt.ylabel('c-value')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    

    # c值在2035和2050年的分布
    plt.subplot(3, 1, 3)
    years_to_plot = [2035, 2050]
    c_data = [predictions[predictions['year'] == year]['c_pred'] for year in years_to_plot]
    plt.boxplot(c_data, labels=['2035', '2050'])
    plt.title('Distribution of c-values in 2035 and 2050')
    plt.ylabel('c-value')
    
    plt.tight_layout()
    plt.show()

# 主函数
def main():
    try:
        print("Using existing processed data")
        
        # 3. 模拟分析
        df_sim, launched_data, non_launched_data = simulation_analysis(df)
        print("Simulation analysis completed")
        
        # 4. 预测分析
        predictions = forecast_analysis(launched_data)
        print("Forecast analysis completed")
        
        # 5. 输出结果
        print("\nPrediction Summary:")
        print("\nYear 2035:")
        pred_2035 = predictions[predictions['year'] == 2035]
        print("\nc-value statistics:")
        print(pred_2035['c_pred'].describe())
      
        
        print("\nYear 2050:")
        pred_2050 = predictions[predictions['year'] == 2050]
        print("\nc-value statistics:")
        print(pred_2050['c_pred'].describe())
 
        
        # 6. 可视化结果
        plot_results(df_sim, predictions)
        
        # 7. 保存结果到CSV文件
        df_sim.to_csv('simulation_results.csv', index=False)
        predictions.to_csv('predictions_results.csv', index=False)
        
        # 8. 输出详细数据
        print("\n模拟结果数据预览（前5行）：")
        print(df_sim.head())
        print("\n预测结果数据预览（前5行）：")
        print(predictions.head())
        
        # 9. 输出基本统计信息
        print("\n模拟结果基本统计：")
        print(df_sim.describe())
        print("\n预测结果基本统计：")
        print(predictions.describe())
        
        return df_sim, predictions
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None, None

# 运行分析a
df_result, predictions = main()

In [ ]:
# 在main()运行之后添加以下代码

def calculate_equilibrium_parameters(countries, tau=0.0000062, gamma=0.00001):
    """
    计算均衡所需的固定参数
    """
    # 计算每个国家的αi（由于gamma固定，所有国家的αi相同）
    alpha = 1 / (tau + gamma)
    alphas = {country: alpha for country in countries}
    
    # 计算分母项 1/τ + Σαj
    n = len(countries)
    denominator = (1/tau) + (n * alpha)
    
    # 计算每个国家的λi（由于alpha相同，所有国家的λi也相同）
    lambda_i = alpha / denominator
    lambdas = {country: lambda_i for country in countries}
    
    print(f"\n参数计算过程:")
    print(f"tau = {tau}, gamma = {gamma}")
    print(f"alpha = 1/({tau} + {gamma}) = {alpha}")
    print(f"国家数量 n = {n}")
    print(f"分母 = 1/{tau} + {n} * {alpha} = {denominator}")
    print(f"lambda = {alpha} / {denominator} = {lambda_i}")
    
    return alphas, lambdas

def calculate_launches(predictions, alphas, lambdas, df, tau=0.0000062):
    """
    根据预测的c值计算每个国家的发射量
    xit = αit[(1-cit) - Σj∈N λjt(1-cjt) - τSt-1]
    """
    new_predictions = predictions.copy()
    new_predictions['theoretical_flow_launch'] = 0.0
    new_predictions['sum_lambda_c'] = 0.0
    
    # 获取最后一年的prev_total_launch作为初始St-1
    last_year_data = df[df['year'] == df['year'].max()]
    prev_total_launch = 18004.742
    
    for year in sorted(new_predictions['year'].unique()):
        year_data = new_predictions[new_predictions['year'] == year]
        
        # 计算当年的Σλj(1-cj)
        sum_lambda_c = 0
        for _, row in year_data.iterrows():
            country = row['country']
            c_j = row['c_pred']
            term = lambdas[country] * (1 - c_j)
            sum_lambda_c += term
        
        # 将sum_lambda_c值添加到该年份的所有行
        new_predictions.loc[new_predictions['year'] == year, 'sum_lambda_c'] = sum_lambda_c
        
        # 计算每个国家的发射量
        total_launches = 0
        for country in year_data['country'].unique():
            country_mask = (new_predictions['year'] == year) & (new_predictions['country'] == country)
            c_i = new_predictions.loc[country_mask, 'c_pred'].iloc[0]
            # 加入τSt-1项
            x_i = alphas[country] * ((1 - c_i) - sum_lambda_c - tau * prev_total_launch)
            x_i = max(x_i, 0)
            new_predictions.loc[country_mask, 'theoretical_flow_launch'] = x_i
            total_launches += x_i
        
        # 更新St-1为当年的总发射量，用于下一年的计算
        prev_total_launch += total_launches
        
        # 添加当年的total_launch到预测结果中
        new_predictions.loc[new_predictions['year'] == year, 'total_launch_t_minus_1'] = prev_total_launch
    
    return new_predictions

# 主要计算过程
countries = predictions['country'].unique()
print(f"\n总计{len(countries)}个国家参与计算")

# 计算均衡参数
alphas, lambdas = calculate_equilibrium_parameters(countries)

# 保存alpha和lambda值
alpha_lambda_df = pd.DataFrame({
    'country': list(alphas.keys()),
    'alpha': list(alphas.values()),
    'lambda': list(lambdas.values())
})
alpha_lambda_df.to_csv('alpha_lambda_values.csv', index=False)
print("\nAlpha和Lambda值已保存到 'alpha_lambda_values.csv'")

# 计算发射量
new_predictions = calculate_launches(predictions, alphas, lambdas, df)
# 输出结果统计
print("\n基于理论公式的发射量预测统计:")
print("\n2035年:")
pred_2035 = new_predictions[new_predictions['year'] == 2035]
print(pred_2035[['country', 'c_pred', 'theoretical_flow_launch']].to_string())

print("\n2050年:")
pred_2050 = new_predictions[new_predictions['year'] == 2050]
print(pred_2050[['country', 'c_pred', 'theoretical_flow_launch']].to_string())

# 保存新的预测结果
new_predictions.to_csv('theoretical_predictions_results.csv', index=False)
print("\n新的预测结果已保存到 'theoretical_predictions_results.csv'")

In [ ]:
# 导入基础数据
df = pd.read_csv('simulation_results.csv')

# 删除指定的列
columns_to_drop = ['c_hat', 'gamma_hat', 'gamma_hat_constrained', 
                   'c_hat_constrained', 'LHS_pred']
df = df.drop(columns=columns_to_drop)

# 计算每年的cost_global（基于flow_launch=0的样本）
cost_global_by_year = df[df['flow_launch'] == 0].groupby('year')['c_sim'].first()

# 为每个样本添加对应年份的cost_global
df['cost_global'] = df['year'].map(cost_global_by_year)

# 生成launch_status变量
df['launch_status'] = (df['flow_launch'] > 0).astype(int)

# 保存新的数据集
df.to_csv('simulation_latestlaunch.csv', index=False)

# 输出处理结果摘要
print("数据处理完成：")
print(f"总样本数: {len(df)}")
print("\n前5行数据预览：")
print(df.head())
print("\n每年的cost_global值：")
print(cost_global_by_year)
print("\nlaunch_status的分布：")
print(df['launch_status'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate launch frequency between 2015-2024
launch_freq = df[
    (df['year'] >= 2015) & 
    (df['year'] <= 2024)
].groupby('country')['launch_status'].sum().reset_index()
launch_freq.columns = ['country', 'frequency_launch']

# Merge frequency info with 2024 data
df_2024 = df[df['year'] == 2024].copy()
df_2024 = df_2024.merge(launch_freq, on='country', how='left')

# Create figure
plt.figure(figsize=(15, 6))

# Subplot 1: Distribution for all countries
plt.subplot(1, 2, 1)
plt.hist(df_2024['frequency_launch'], bins=20, edgecolor='black')
plt.title('Launch Frequency Distribution - All Countries (2015-2024)')
plt.xlabel('Launch Frequency')
plt.ylabel('Number of Countries')

# Add descriptive statistics
stats_all = df_2024['frequency_launch'].describe()
plt.text(0.7, 0.95, 
         f'Mean: {stats_all["mean"]:.2f}\nStd: {stats_all["std"]:.2f}\n' + 
         f'Min: {stats_all["min"]:.0f}\nMax: {stats_all["max"]:.0f}\n' +
         f'Total Countries: {len(df_2024)}',
         transform=plt.gca().transAxes,
         bbox=dict(facecolor='white', alpha=0.8))

# Subplot 2: Distribution for active countries only
df_2024_active = df_2024[df_2024['frequency_launch'] > 0]
plt.subplot(1, 2, 2)
plt.hist(df_2024_active['frequency_launch'], bins=20, edgecolor='black')
plt.title('Launch Frequency Distribution - Active Countries (2015-2024)')
plt.xlabel('Launch Frequency')
plt.ylabel('Number of Countries')

# Add descriptive statistics
stats_active = df_2024_active['frequency_launch'].describe()
plt.text(0.7, 0.95, 
         f'Mean: {stats_active["mean"]:.2f}\nStd: {stats_active["std"]:.2f}\n' + 
         f'Min: {stats_active["min"]:.0f}\nMax: {stats_active["max"]:.0f}\n' +
         f'Total Countries: {len(df_2024_active)}',
         transform=plt.gca().transAxes,
         bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

# Print detailed statistics
print("\nLaunch Frequency Statistics - All Countries:")
print(stats_all)
print("\nLaunch Frequency Statistics - Active Countries:")
print(stats_active)

# Save frequency data
launch_freq.to_csv('launch_frequency_2015_2024.csv', index=False)
print("\nLaunch frequency data saved to 'launch_frequency_2015_2024.csv'")

# Print top 10 most active countries
print("\nTop 10 Countries by Launch Frequency:")
print(launch_freq.nlargest(10, 'frequency_launch'))

In [ ]:
# 筛选发射频率大于等于5的国家
major_players = launch_freq[launch_freq['frequency_launch'] >= 5]

# 打印主要参与者信息
print("Major Players (Launch Frequency >= 5):")
print(f"Total number of major players: {len(major_players)}")
print("\nDetailed list of major players:")
print(major_players.sort_values('frequency_launch', ascending=False))

# 使用这些主要参与者筛选原始数据
major_players_list = major_players['country'].tolist()
df_major = df[df['country'].isin(major_players_list)].copy()

# 打印数据集信息
print(f"\nOriginal dataset size: {len(df)}")
print(f"Major players dataset size: {len(df_major)}")
print(f"Percentage of data retained: {(len(df_major)/len(df)*100):.2f}%")

# 保存主要参与者名单
major_players.to_csv('major_players_list.csv', index=False)
print("\nMajor players list saved to 'major_players_list.csv'")

# 保存主要参与者的完整数据
df_major.to_csv('major_players_data.csv', index=False)
print("Major players complete data saved to 'major_players_data.csv'")

# 显示每个主要参与者的发射频率分布
plt.figure(figsize=(12, 6))
plt.bar(range(len(major_players)), major_players['frequency_launch'])
plt.xticks(range(len(major_players)), major_players['country'], rotation=45, ha='right')
plt.title('Launch Frequency Distribution Among Major Players (2015-2024)')
plt.xlabel('Country')
plt.ylabel('Launch Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# 读取major players数据
df_major = pd.read_csv('major_players_data.csv')

def simulation_analysis_major(df):
    """
    针对主要参与者的模拟分析函数
    """
    # 创建新的预测结果列
    df['gamma_fixed'] = 0.00001  # 固定gamma值
    df['c_sim'] = df['LHS'] - (0.00001 * df['flow_launch'])
    df['LHS_pred'] = df['c_sim'] + (0.00001 * df['flow_launch'])
    
    return df

def forecast_analysis_major(df):
    """
    针对主要参与者的预测分析函数
    """
    current_year = df['year'].max()
    years_to_2050 = 2050 - current_year
    predictions = pd.DataFrame()
    arima_orders = {}  # 用于存储每个国家的ARIMA阶数
    
    for country in df['country'].unique():
        country_data = df[df['country'] == country]
        
        try:
            c_series = country_data['c_sim']
            
            if len(c_series) > 5:
                order = find_best_arima_order(c_series)
            else:
                order = (1,0,0)
            
            arima_orders[country] = order
            
            model = ARIMA(c_series, order=order)
            results = model.fit()
            c_forecast = results.forecast(steps=years_to_2050)
            c_forecast = np.maximum(c_forecast, 1e-10)
            
            country_predictions = pd.DataFrame({
                'year': range(current_year + 1, 2051),
                'country': country,
                'c_pred': c_forecast,
                'arima_order': str(order)
            })
            
            last_lhs = country_data['LHS'].iloc[-1]
            country_predictions['flow_launch_pred'] = np.maximum(
                (last_lhs - country_predictions['c_pred']) / 0.00001, 
                0
            )
            
            predictions = pd.concat([predictions, country_predictions])
            
        except Exception as e:
            print(f"预测{country}时出错: {str(e)}")
            continue
    
    print("\n主要参与者的ARIMA模型阶数:")
    for country, order in arima_orders.items():
        print(f"{country}: ARIMA{order}")
    
    return predictions

def main_major_players():
    try:
        print("使用主要参与者数据进行分析")
        
        # 模拟分析
        df_sim = simulation_analysis_major(df_major)
        print("主要参与者模拟分析完成")
        
        # 预测分析
        predictions = forecast_analysis_major(df_sim)
        print("主要参与者预测分析完成")
        
        # 输出结果
        print("\n预测结果摘要:")
        print("\n2035年:")
        pred_2035 = predictions[predictions['year'] == 2035]
        print("\nc值统计:")
        print(pred_2035['c_pred'].describe())
        print("\n发射量统计:")
        print(pred_2035['flow_launch_pred'].describe())
        
        print("\n2050年:")
        pred_2050 = predictions[predictions['year'] == 2050]
        print("\nc值统计:")
        print(pred_2050['c_pred'].describe())
        print("\n发射量统计:")
        print(pred_2050['flow_launch_pred'].describe())
        
        # 可视化结果
        plot_results(df_sim, predictions)
        
        # 保存结果
        df_sim.to_csv('major_players_simulation_results.csv', index=False)
        predictions.to_csv('major_players_predictions_results.csv', index=False)
        
        print("\n结果已保存到 'major_players_simulation_results.csv' 和 'major_players_predictions_results.csv'")
        
        return df_sim, predictions
        
    except Exception as e:
        print(f"发生错误: {str(e)}")
        return None, None

# 运行主要参与者分析
df_major_result, major_predictions = main_major_players()

In [ ]:
# 使用major_players的预测结果进行均衡计算

# 读取预测数据
major_predictions = pd.read_csv('major_players_predictions_results.csv')

# 计算主要参与者的均衡参数
major_countries = major_predictions['country'].unique()
print(f"\n总计{len(major_countries)}个主要参与国家参与计算")

# 计算均衡参数 (使用相同的tau和gamma值)
major_alphas, major_lambdas = calculate_equilibrium_parameters(major_countries, tau=0.0000062, gamma=0.00001)

# 保存主要参与者的alpha和lambda值
major_alpha_lambda_df = pd.DataFrame({
    'country': list(major_alphas.keys()),
    'alpha': list(major_alphas.values()),
    'lambda': list(major_lambdas.values())
})
major_alpha_lambda_df.to_csv('major_players_alpha_lambda_values.csv', index=False)
print("\n主要参与者的Alpha和Lambda值已保存到 'major_players_alpha_lambda_values.csv'")

# 计算主要参与者的理论发射量
major_theoretical_predictions = calculate_launches(major_predictions, major_alphas, major_lambdas, df)

# 输出主要参与者的结果统计
print("\n主要参与者基于理论公式的发射量预测统计:")
print("\n2035年:")
major_pred_2035 = major_theoretical_predictions[major_theoretical_predictions['year'] == 2035]
print(major_pred_2035[['country', 'c_pred', 'theoretical_flow_launch']].to_string())

print("\n2050年:")
major_pred_2050 = major_theoretical_predictions[major_theoretical_predictions['year'] == 2050]
print(major_pred_2050[['country', 'c_pred', 'theoretical_flow_launch']].to_string())

# 保存主要参与者的新预测结果
major_theoretical_predictions.to_csv('major_players_theoretical_predictions_results.csv', index=False)
print("\n主要参与者的新预测结果已保存到 'major_players_theoretical_predictions_results.csv'")

# 添加结果比较
print("\n预测结果比较分析:")
for year in [2035, 2050]:
    year_data = major_theoretical_predictions[major_theoretical_predictions['year'] == year]
    print(f"\n{year}年预测统计:")
    print("\n原始预测发射量统计:")
    print(year_data['flow_launch_pred'].describe())
    print("\n理论模型发射量统计:")
    print(year_data['theoretical_flow_launch'].describe())

In [ ]:
# 新的代码单元：从头计算c_sim并基于新gamma进行预测
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# 读取原始数据
df_original = pd.read_csv('major_players_data.csv')

# 设置要测试的gamma值
test_gamma = 0.0001  # 大幅降低的gamma值
tau = 0.000003  # 保持原始tau不变

# 1. 重新计算c_sim
df = df_original.copy()
df['gamma_test'] = test_gamma  # 使用新的gamma值
df['c_sim_new'] = df['LHS'] - (test_gamma * df['flow_launch'])  # 使用新gamma计算c_sim

# 2. 基于新c_sim重新进行ARIMA预测
def find_best_arima_order(series, max_p=3, max_d=2, max_q=3):
    """为时间序列找到最优的ARIMA模型阶数"""
    adf_result = adfuller(series)
    d = 1 if adf_result[1] > 0.05 else 0
    
    best_aic = float('inf')
    best_order_aic = None
    
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                model = ARIMA(series, order=(p, d, q))
                results = model.fit()
                if results.aic < best_aic:
                    best_aic = results.aic
                    best_order_aic = (p, d, q)
            except:
                continue
    
    return best_order_aic if best_order_aic else (1,0,0)

# 准备新的预测
current_year = df['year'].max()
years_to_2050 = 2050 - current_year
predictions_new = pd.DataFrame()
arima_orders = {}

for country in df['country'].unique():
    country_data = df[df['country'] == country]
    
    try:
        c_series = country_data['c_sim_new']
        
        if len(c_series) > 5:
            order = find_best_arima_order(c_series)
        else:
            order = (1,0,0)
        
        arima_orders[country] = order
        
        model = ARIMA(c_series, order=order)
        results = model.fit()
        c_forecast = results.forecast(steps=years_to_2050)
        c_forecast = np.maximum(c_forecast, 1e-10)  # 防止负值
        
        country_predictions = pd.DataFrame({
            'year': range(current_year + 1, 2051),
            'country': country,
            'c_pred': c_forecast,
            'arima_order': str(order)
        })
        
        last_lhs = country_data['LHS'].iloc[-1]
        country_predictions['flow_launch_pred'] = np.maximum(
            (last_lhs - country_predictions['c_pred']) / test_gamma, 
            0
        )
        
        predictions_new = pd.concat([predictions_new, country_predictions])
        
    except Exception as e:
        print(f"预测{country}时出错: {str(e)}")
        continue

print("\n使用新gamma值的ARIMA模型阶数:")
for country, order in arima_orders.items():
    print(f"{country}: ARIMA{order}")

# 3. 使用相同的gamma值计算均衡
countries = predictions_new['country'].unique()

# 计算alpha和lambda值
alphas = {}
lambdas = {}

for country in countries:
    alphas[country] = 1 / (tau + test_gamma)

alpha_sum = sum(alphas.values())

for country in countries:
    lambdas[country] = alphas[country] / ((1/tau) + alpha_sum)

# 计算发射量
predictions_new['theoretical_flow_launch'] = 0.0
predictions_new['sum_lambda_c'] = 0.0

# 获取最后一年的total_launch作为初始St-1
prev_total_launch = 18004.742  # 使用原始值

active_countries_by_year = {}

for year in sorted(predictions_new['year'].unique()):
    year_data = predictions_new[predictions_new['year'] == year]
    
    # 计算当年的Σλj(1-cj)
    sum_lambda_c = 0
    for _, row in year_data.iterrows():
        country = row['country']
        c_j = row['c_pred']
        term = lambdas[country] * (1 - c_j)
        sum_lambda_c += term
    
    predictions_new.loc[predictions_new['year'] == year, 'sum_lambda_c'] = sum_lambda_c
    
    active_countries = 0
    total_launches = 0
    
    for country in year_data['country'].unique():
        country_mask = (predictions_new['year'] == year) & (predictions_new['country'] == country)
        c_i = predictions_new.loc[country_mask, 'c_pred'].iloc[0]
        
        x_i = alphas[country] * ((1 - c_i) - sum_lambda_c - tau * prev_total_launch)
        x_i = max(x_i, 0)
        
        predictions_new.loc[country_mask, 'theoretical_flow_launch'] = x_i
        
        if x_i > 0.1:
            active_countries += 1
        
        total_launches += x_i
    
    prev_total_launch += total_launches
    predictions_new.loc[predictions_new['year'] == year, 'total_launch_t_minus_1'] = prev_total_launch
    active_countries_by_year[year] = active_countries

# 保存结果
predictions_new.to_csv(f'consistent_gamma_{test_gamma:.6f}_predictions.csv', index=False)

# 统计和可视化结果
print(f"\n使用一致的gamma={test_gamma}的预测结果:")
print(f"2035年有发射的国家数量: {active_countries_by_year.get(2035, 0)}")
print(f"2050年有发射的国家数量: {active_countries_by_year.get(2050, 0)}")
print(f"2035年总发射量: {predictions_new[predictions_new['year']==2035]['theoretical_flow_launch'].sum():.2f}")
print(f"2050年总发射量: {predictions_new[predictions_new['year']==2050]['theoretical_flow_launch'].sum():.2f}")

# 显示2035年发射量最大的国家
top_countries_2035 = predictions_new[predictions_new['year']==2035].nlargest(10, 'theoretical_flow_launch')
print("\n2035年发射量最大的10个国家:")
for _, row in top_countries_2035.iterrows():
    if row['theoretical_flow_launch'] > 0:
        print(f"{row['country']}: {row['theoretical_flow_launch']:.2f}")

# 创建2035年有发射的国家柱状图
active_2035 = predictions_new[(predictions_new['year']==2035) & (predictions_new['theoretical_flow_launch'] > 0)]
active_2035 = active_2035.sort_values('theoretical_flow_launch', ascending=False)

plt.figure(figsize=(14, 6))
plt.bar(active_2035['country'], active_2035['theoretical_flow_launch'])
plt.title(f'2035年各国发射量预测 (gamma={test_gamma})')
plt.xlabel('国家')
plt.ylabel('发射量')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'countries_launches_2035_gamma_{test_gamma:.6f}.png', dpi=300)
plt.show()

In [ ]:
# 新的代码单元：测试不同的gamma和tau组合
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import itertools
import os

# 读取原始数据
df_original = pd.read_csv('major_players_data.csv')

# 设置要测试的gamma值和tau值
gamma_values = [0.00009,0.000095,0.0001]
tau_values = [ 0.000002,0.00000225,0.0000025,0.00000275,0.000003]

# 创建存储有效组合结果的列表
valid_combinations = []
valid_predictions = []

# 为每个组合创建一个目录来存储结果
results_dir = "gamma_tau_results"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

def find_best_arima_order(series, max_p=3, max_d=2, max_q=3):
    """为时间序列找到最优的ARIMA模型阶数"""
    adf_result = adfuller(series)
    d = 1 if adf_result[1] > 0.05 else 0
    
    best_aic = float('inf')
    best_order_aic = None
    
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                model = ARIMA(series, order=(p, d, q))
                results = model.fit()
                if results.aic < best_aic:
                    best_aic = results.aic
                    best_order_aic = (p, d, q)
            except:
                continue
    
    return best_order_aic if best_order_aic else (1,0,0)

# 遍历所有gamma和tau的组合
for gamma, tau in itertools.product(gamma_values, tau_values):
    try:
        print(f"\n测试组合: gamma={gamma}, tau={tau}")
        
        # 1. 重新计算c_sim
        df = df_original.copy()
        df['gamma_test'] = gamma
        df['tau_test'] = tau
        df['c_sim_new'] = df['LHS'] - (gamma * df['flow_launch'])
        
        # 2. 基于新c_sim重新进行ARIMA预测
        current_year = df['year'].max()
        years_to_2050 = 2050 - current_year
        predictions_new = pd.DataFrame()
        arima_orders = {}
        
        for country in df['country'].unique():
            country_data = df[df['country'] == country]
            
            try:
                c_series = country_data['c_sim_new']
                
                if len(c_series) > 5:
                    order = find_best_arima_order(c_series)
                else:
                    order = (1,0,0)
                
                arima_orders[country] = order
                
                model = ARIMA(c_series, order=order)
                results = model.fit()
                c_forecast = results.forecast(steps=years_to_2050)
                c_forecast = np.maximum(c_forecast, 1e-10)  # 防止负值
                
                country_predictions = pd.DataFrame({
                    'year': range(current_year + 1, 2051),
                    'country': country,
                    'c_pred': c_forecast,
                    'arima_order': str(order),
                    'gamma': gamma,
                    'tau': tau
                })
                
                last_lhs = country_data['LHS'].iloc[-1]
                country_predictions['flow_launch_pred'] = np.maximum(
                    (last_lhs - country_predictions['c_pred']) / gamma, 
                    0
                )
                
                predictions_new = pd.concat([predictions_new, country_predictions])
                
            except Exception as e:
                print(f"预测{country}时出错: {str(e)}")
                continue
        
        # 3. 使用相同的gamma和tau值计算均衡
        countries = predictions_new['country'].unique()
        
        # 计算alpha和lambda值
        alphas = {}
        lambdas = {}
        
        for country in countries:
            alphas[country] = 1 / (tau + gamma)
        
        alpha_sum = sum(alphas.values())
        
        for country in countries:
            lambdas[country] = alphas[country] / ((1/tau) + alpha_sum)
        
        # 计算发射量
        predictions_new['theoretical_flow_launch'] = 0.0
        predictions_new['sum_lambda_c'] = 0.0
        
        # 获取最后一年的total_launch作为初始St-1
        prev_total_launch = 18004.742  # 使用原始值
        
        active_countries_by_year = {}
        
        for year in sorted(predictions_new['year'].unique()):
            year_data = predictions_new[predictions_new['year'] == year]
            
            # 计算当年的Σλj(1-cj)
            sum_lambda_c = 0
            for _, row in year_data.iterrows():
                country = row['country']
                c_j = row['c_pred']
                term = lambdas[country] * (1 - c_j)
                sum_lambda_c += term
            
            predictions_new.loc[predictions_new['year'] == year, 'sum_lambda_c'] = sum_lambda_c
            
            active_countries = 0
            total_launches = 0
            
            for country in year_data['country'].unique():
                country_mask = (predictions_new['year'] == year) & (predictions_new['country'] == country)
                c_i = predictions_new.loc[country_mask, 'c_pred'].iloc[0]
                
                x_i = alphas[country] * ((1 - c_i) - sum_lambda_c - tau * prev_total_launch)
                x_i = max(x_i, 0)
                
                predictions_new.loc[country_mask, 'theoretical_flow_launch'] = x_i
                
                if x_i > 0.1:
                    active_countries += 1
                
                total_launches += x_i
            
            prev_total_launch += total_launches
            predictions_new.loc[predictions_new['year'] == year, 'total_launch_t_minus_1'] = prev_total_launch
            active_countries_by_year[year] = active_countries
        
        # 保存结果到文件
        file_name = f"{results_dir}/gamma_{gamma:.6f}_tau_{tau:.6f}_predictions.csv"
        predictions_new.to_csv(file_name, index=False)
        
       # ... 现有代码 ...

        # 检查筛选条件
        # 1. 检查美国的c_pred是否有小于0.01的情况
        us_data = predictions_new[predictions_new['country'] == 'United States']
        min_c_pred_us = us_data['c_pred'].min()
        
        # 2. 检查美国在2025年的theoretical_flow_launch是否在2000到4000之间
        us_2025 = us_data[us_data['year'] == 2025]
        us_2025_launch = 0
        if not us_2025.empty:
            us_2025_launch = us_2025['theoretical_flow_launch'].values[0]
        
        # 3. 检查中国在2025年的theoretical_flow_launch是否在250到500之间
        china_data = predictions_new[predictions_new['country'] == 'China']
        china_2025 = china_data[china_data['year'] == 2025]
        china_2025_launch = 0
        if not china_2025.empty:
            china_2025_launch = china_2025['theoretical_flow_launch'].values[0]
            
        # 4. 检查CIS在2025年的theoretical_flow_launch是否在50到200之间
        cis_data = predictions_new[predictions_new['country'] == 'Commonwealth of Independent States']
        cis_2025 = cis_data[cis_data['year'] == 2025]
        cis_2025_launch = 0
        if not cis_2025.empty:
            cis_2025_launch = cis_2025['theoretical_flow_launch'].values[0]
        
        # 输出筛选结果
        print(f"美国最小c_pred: {min_c_pred_us:.6f}")
        print(f"美国2025年发射量: {us_2025_launch:.2f}")
        print(f"中国2025年发射量: {china_2025_launch:.2f}")
        print(f"CIS 2025年发射量: {cis_2025_launch:.2f}")
        
        # 如果满足筛选条件，则将该组合添加到有效组合列表
        if (min_c_pred_us >= 0.01 and 
            2000 <= us_2025_launch < 4000 and
            250 <= china_2025_launch < 500 and
            50 <= cis_2025_launch < 400):
            print(f"组合 gamma={gamma}, tau={tau} 满足筛选条件")
            valid_combinations.append((gamma, tau))
            valid_predictions.append(predictions_new)
        else:
            print(f"组合 gamma={gamma}, tau={tau} 不满足筛选条件")
            
# ... 现有代码 ...
            
    except Exception as e:
        print(f"处理组合 gamma={gamma}, tau={tau} 时出错: {str(e)}")
        continue

# 如果有有效组合，则合并结果
if valid_combinations:
    print(f"\n找到 {len(valid_combinations)} 个有效组合:")
    for gamma, tau in valid_combinations:
        print(f"gamma={gamma}, tau={tau}")
    
    # 合并所有有效预测结果
    combined_results = pd.DataFrame()
    
    for i, (gamma, tau) in enumerate(valid_combinations):
        pred_df = valid_predictions[i]
        
        # 如果是第一个有效组合，保留基本列
        if i == 0:
            combined_results = pred_df[['year', 'country']].copy()
        
        # 添加特定组合的c_pred和theoretical_flow_launch列
        gamma_tau_suffix = f"g{gamma:.6f}_t{tau:.6f}"
        combined_results[f'c_pred_{gamma_tau_suffix}'] = pred_df['c_pred']
        combined_results[f'theoretical_flow_launch_{gamma_tau_suffix}'] = pred_df['theoretical_flow_launch']
    
    # 保存合并结果
    combined_results.to_csv(f"{results_dir}/combined_valid_predictions.csv", index=False)
    print(f"\n合并结果已保存到 {results_dir}/combined_valid_predictions.csv")
    
    # 创建美国数据的单独表格
    us_results = combined_results[combined_results['country'] == 'United States'].copy()
    us_results.to_csv(f"{results_dir}/us_valid_predictions.csv", index=False)
    print(f"美国数据已保存到 {results_dir}/us_valid_predictions.csv")
    
    # 创建2025年数据的单独表格
    year_2025_results = combined_results[combined_results['year'] == 2025].copy()
    year_2025_results.to_csv(f"{results_dir}/year_2025_valid_predictions.csv", index=False)
    print(f"2025年数据已保存到 {results_dir}/year_2025_valid_predictions.csv")
    
    # 为每个有效组合创建2025年发射量的柱状图
    for gamma, tau in valid_combinations:
        gamma_tau_suffix = f"g{gamma:.6f}_t{tau:.6f}"
        year_2025 = combined_results[combined_results['year'] == 2025].copy()
        year_2025 = year_2025.sort_values(f'theoretical_flow_launch_{gamma_tau_suffix}', ascending=False)
        top_countries = year_2025.head(10)
        
        plt.figure(figsize=(14, 6))
        plt.bar(top_countries['country'], top_countries[f'theoretical_flow_launch_{gamma_tau_suffix}'])
        plt.title(f'2025年各国发射量预测 (gamma={gamma}, tau={tau})')
        plt.xlabel('国家')
        plt.ylabel('发射量')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(f"{results_dir}/top_countries_2025_gamma_{gamma:.6f}_tau_{tau:.6f}.png", dpi=300)
        plt.close()
else:
    print("\n没有找到满足筛选条件的组合")

In [ ]:
# 模型准确性测试：对比2024年预测值与实际值
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 自定义MAPE函数（因为sklearn版本较旧）
def mean_absolute_percentage_error(y_true, y_pred):
    """
    计算平均绝对百分比误差
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # 避免除以零
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# 创建结果目录
results_dir = "model_accuracy_test"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

# 读取原始数据
df_original = pd.read_csv('major_players_data.csv')

# 2024年的实际累计发射量
X_total_cum_2024 = 20807.979

# 设置要测试的gamma和tau组合
param_combinations = [
    (0.0001, 2.25e-06),
    (0.0001, 2.5e-06)
]

# 定义ARIMA模型阶数选择函数
def find_best_arima_order(series, max_p=3, max_d=2, max_q=3):
    """为时间序列找到最优的ARIMA模型阶数"""
    adf_result = adfuller(series)
    d = 1 if adf_result[1] > 0.05 else 0
    
    best_aic = float('inf')
    best_order_aic = None
    
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                model = ARIMA(series, order=(p, d, q))
                results = model.fit()
                if results.aic < best_aic:
                    best_aic = results.aic
                    best_order_aic = (p, d, q)
            except:
                continue
    
    return best_order_aic if best_order_aic else (1,0,0)

# 筛选2023年及之前的数据
df_train = df_original[df_original['year'] <= 2023].copy()
df_2024 = df_original[df_original['year'] == 2024].copy()

# 存储所有预测结果的DataFrame
all_predictions = pd.DataFrame()

# 方法1和2：使用ARIMA模型和不同的gamma/tau组合
for method_idx, (gamma, tau) in enumerate(param_combinations, 1):
    print(f"\n测试方法{method_idx}: ARIMA模型 (gamma={gamma}, tau={tau})")
    
    # 计算c_sim
    df_train['gamma_test'] = gamma
    df_train['tau_test'] = tau
    df_train['c_sim'] = df_train['LHS'] - (gamma * df_train['flow_launch'])
    
    # 基于c_sim进行ARIMA预测
    predictions = pd.DataFrame()
    
    for country in df_train['country'].unique():
        country_data = df_train[df_train['country'] == country]
        
        try:
            c_series = country_data['c_sim']
            
            if len(c_series) > 5:
                order = find_best_arima_order(c_series)
            else:
                order = (1,0,0)
            
            model = ARIMA(c_series, order=order)
            results = model.fit()
            c_forecast = results.forecast(steps=1)  # 预测2024
            c_forecast = np.maximum(c_forecast, 1e-10)  # 防止负值
            
            country_predictions = pd.DataFrame({
                'year': [2024],
                'country': country,
                'c_pred': c_forecast,
                'arima_order': str(order),
                'method': f'ARIMA_g{gamma}_t{tau}',
                'gamma': gamma,
                'tau': tau
            })
            
            last_lhs = country_data['LHS'].iloc[-1]
            country_predictions['flow_launch_pred'] = np.maximum(
                (last_lhs - country_predictions['c_pred']) / gamma, 
                0
            )
            
            predictions = pd.concat([predictions, country_predictions])
            
        except Exception as e:
            print(f"预测{country}时出错: {str(e)}")
            continue
    
    # 计算均衡发射量
    countries = predictions['country'].unique()
    
    # 计算alpha和lambda值
    alphas = {}
    lambdas = {}
    
    for country in countries:
        alphas[country] = 1 / (tau + gamma)
    
    alpha_sum = sum(alphas.values())
    
    for country in countries:
        lambdas[country] = alphas[country] / ((1/tau) + alpha_sum)
    
    # 计算发射量
    predictions['theoretical_flow_launch'] = 0.0
    predictions['sum_lambda_c'] = 0.0
    
    # 获取2023年的total_launch作为初始St-1
    prev_total_launch = df_train[df_train['year'] == 2023]['flow_launch'].sum()
    
    for year in sorted(predictions['year'].unique()):
        year_data = predictions[predictions['year'] == year]
        
        # 计算当年的Σλj(1-cj)
        sum_lambda_c = 0
        for _, row in year_data.iterrows():
            country = row['country']
            c_j = row['c_pred']
            term = lambdas[country] * (1 - c_j)
            sum_lambda_c += term
        
        predictions.loc[predictions['year'] == year, 'sum_lambda_c'] = sum_lambda_c
        
        total_launches = 0
        
        for country in year_data['country'].unique():
            country_mask = (predictions['year'] == year) & (predictions['country'] == country)
            c_i = predictions.loc[country_mask, 'c_pred'].iloc[0]
            
            x_i = alphas[country] * ((1 - c_i) - sum_lambda_c - tau * prev_total_launch)
            x_i = max(x_i, 0)
            
            predictions.loc[country_mask, 'theoretical_flow_launch'] = x_i
            total_launches += x_i
        
        prev_total_launch += total_launches
        predictions.loc[predictions['year'] == year, 'total_launch_t_minus_1'] = prev_total_launch
    
    # 将结果添加到总预测结果中
    all_predictions = pd.concat([all_predictions, predictions])

# 方法3：使用线性回归预测c_sim
print("\n测试方法3: 线性回归模型")

# 选择最优的gamma和tau组合（这里使用第一个组合）
gamma, tau = param_combinations[0]

# 计算c_sim
df_train['gamma_test'] = gamma
df_train['tau_test'] = tau
df_train['c_sim'] = df_train['LHS'] - (gamma * df_train['flow_launch'])

# 使用线性回归预测c_sim
linear_predictions = pd.DataFrame()

for country in df_train['country'].unique():
    country_data = df_train[df_train['country'] == country]
    
    try:
        # 准备线性回归数据
        X = country_data['year'].values.reshape(-1, 1)
        y = country_data['c_sim'].values
        
        if len(y) > 1:  # 至少需要两个点进行线性回归
            model = LinearRegression()
            model.fit(X, y)
            
            # 预测2024和2025年
            future_years = np.array([2024).reshape(-1, 1)
            c_forecast = model.predict(future_years)
            c_forecast = np.maximum(c_forecast, 1e-10)  # 防止负值
            
            country_predictions = pd.DataFrame({
                'year': [2024],
                'country': country,
                'c_pred': c_forecast,
                'method': 'Linear_Regression',
                'gamma': gamma,
                'tau': tau
            })
            
            last_lhs = country_data['LHS'].iloc[-1]
            country_predictions['flow_launch_pred'] = np.maximum(
                (last_lhs - country_predictions['c_pred']) / gamma, 
                0
            )
            
            linear_predictions = pd.concat([linear_predictions, country_predictions])
            
        else:
            print(f"{country}数据不足，无法进行线性回归")
            
    except Exception as e:
        print(f"预测{country}时出错: {str(e)}")
        continue

# 计算均衡发射量（与ARIMA方法相同的逻辑）
countries = linear_predictions['country'].unique()

# 计算alpha和lambda值
alphas = {}
lambdas = {}

for country in countries:
    alphas[country] = 1 / (tau + gamma)

alpha_sum = sum(alphas.values())

for country in countries:
    lambdas[country] = alphas[country] / ((1/tau) + alpha_sum)

# 计算发射量
linear_predictions['theoretical_flow_launch'] = 0.0
linear_predictions['sum_lambda_c'] = 0.0

# 获取2023年的total_launch作为初始St-1
prev_total_launch = df_train[df_train['year'] == 2023]['flow_launch'].sum()

for year in sorted(linear_predictions['year'].unique()):
    year_data = linear_predictions[linear_predictions['year'] == year]
    
    # 计算当年的Σλj(1-cj)
    sum_lambda_c = 0
    for _, row in year_data.iterrows():
        country = row['country']
        c_j = row['c_pred']
        term = lambdas[country] * (1 - c_j)
        sum_lambda_c += term
    
    linear_predictions.loc[linear_predictions['year'] == year, 'sum_lambda_c'] = sum_lambda_c
    
    total_launches = 0
    
    for country in year_data['country'].unique():
        country_mask = (linear_predictions['year'] == year) & (linear_predictions['country'] == country)
        c_i = linear_predictions.loc[country_mask, 'c_pred'].iloc[0]
        
        x_i = alphas[country] * ((1 - c_i) - sum_lambda_c - tau * prev_total_launch)
        x_i = max(x_i, 0)
        
        linear_predictions.loc[country_mask, 'theoretical_flow_launch'] = x_i
        total_launches += x_i
    
    prev_total_launch += total_launches
    linear_predictions.loc[linear_predictions['year'] == year, 'total_launch_t_minus_1'] = prev_total_launch

# 将线性回归结果添加到总预测结果中
all_predictions = pd.concat([all_predictions, linear_predictions])

# 保存所有预测结果
all_predictions.to_csv(f"{results_dir}/all_predictions.csv", index=False)

# 对比2024年预测与实际值
predictions_2024 = all_predictions[all_predictions['year'] == 2024].copy()

# 合并实际2024年数据
actual_2024 = df_2024[['country', 'flow_launch']].rename(columns={'flow_launch': 'actual_flow_launch'})
comparison = pd.merge(predictions_2024, actual_2024, on='country', how='left')

# 计算差异
comparison['absolute_diff'] = abs(comparison['theoretical_flow_launch'] - comparison['actual_flow_launch'])
# 使用自定义函数计算百分比差异，避免除以零的问题
comparison['percentage_diff'] = 100 * comparison['absolute_diff'] / comparison['actual_flow_launch'].clip(lower=0.1)

# 保存比较结果
comparison.to_csv(f"{results_dir}/prediction_vs_actual_2024.csv", index=False)

# 计算每种方法的平均差异
method_summary = comparison.groupby('method').agg({
    'absolute_diff': 'mean',
    'percentage_diff': 'mean'
}).reset_index()
method_summary.to_csv(f"{results_dir}/method_summary.csv", index=False)

# 重点关注中国、美国和CIS的结果
focus_countries = ['China', 'United States', 'Commonwealth of Independent States']
focus_results = comparison[comparison['country'].isin(focus_countries)].copy()
focus_results = focus_results.sort_values(['country', 'method'])
focus_results.to_csv(f"{results_dir}/focus_countries_comparison.csv", index=False)

# 打印重点国家的结果
print("\n重点国家预测结果对比:")
for country in focus_countries:
    print(f"\n{country}的预测结果:")
    country_data = focus_results[focus_results['country'] == country]
    for _, row in country_data.iterrows():
        print(f"方法: {row['method']}")
        print(f"  预测发射量: {row['theoretical_flow_launch']:.2f}")
        print(f"  实际发射量: {row['actual_flow_launch']:.2f}")
        print(f"  绝对差异: {row['absolute_diff']:.2f}")
        print(f"  百分比差异: {row['percentage_diff']:.2f}%")

# 打印总体结果
print("\n各方法平均差异:")
for _, row in method_summary.iterrows():
    print(f"方法: {row['method']}")
    print(f"  平均绝对差异: {row['absolute_diff']:.2f}")
    print(f"  平均百分比差异: {row['percentage_diff']:.2f}%")

# 创建可视化图表
# 1. 各方法在重点国家的预测对比
for country in focus_countries:
    country_data = focus_results[focus_results['country'] == country]
    
    if not country_data.empty:
        plt.figure(figsize=(15, 8))
        methods = country_data['method'].tolist()
        x = np.arange(len(methods))
        width = 0.35
        
        plt.bar(x - width/2, country_data['theoretical_flow_launch'], width, label='预测值')
        plt.bar(x + width/2, country_data['actual_flow_launch'], width, label='实际值')
        
        plt.xlabel('预测方法') 
        plt.ylabel('发射量')
        plt.title(f'{country} 2024年发射量预测vs实际')
        plt.xticks(x, methods, rotation=45)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{results_dir}/{country}_prediction_comparison.png", dpi=300)
        plt.close()

# 2. 各方法的平均差异对比
plt.figure(figsize=(12, 6))
plt.bar(method_summary['method'], method_summary['percentage_diff'])
plt.xlabel('预测方法')
plt.ylabel('平均百分比差异 (%)')
plt.title('各预测方法的平均百分比差异')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{results_dir}/method_comparison_percentage.png", dpi=300)
plt.close()

print(f"\n所有结果已保存到 {results_dir} 目录")

In [ ]:
# ... existing code ...

# 模型准确性测试：对比2024年预测值与实际值
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 自定义MAPE函数（因为sklearn版本较旧）
def mean_absolute_percentage_error(y_true, y_pred):
    """
    计算平均绝对百分比误差
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # 避免除以零
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# 创建结果目录
results_dir = "model_accuracy_test_2024"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

# 读取原始数据
df_original = pd.read_csv('major_players_data.csv')

# 2024年的实际累计发射量
X_total_cum_2024 = 20807.979

# 设置要测试的gamma和tau组合
gamma_values = [0.0001,0.00014,0.00018,0.00022,0.00026,0.0003]
tau_values = [4e-06,6e-06,8e-06, 1e-05,0.000015]

param_combinations = [(g, t) for g in gamma_values for t in tau_values]

# 定义ARIMA模型阶数选择函数
def find_best_arima_order(series, max_p=3, max_d=2, max_q=3):
    """为时间序列找到最优的ARIMA模型阶数"""
    adf_result = adfuller(series)
    d = 1 if adf_result[1] > 0.05 else 0
    
    best_aic = float('inf')
    best_order_aic = None
    
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                model = ARIMA(series, order=(p, d, q))
                results = model.fit()
                if results.aic < best_aic:
                    best_aic = results.aic
                    best_order_aic = (p, d, q)
            except:
                continue
    
    return best_order_aic if best_order_aic else (1,0,0)

# 筛选2023年及之前的数据
df_train = df_original[df_original['year'] <= 2023].copy()
df_2024 = df_original[df_original['year'] == 2024].copy()

# 存储所有预测结果的DataFrame
all_predictions = pd.DataFrame()

# 方法1：使用ARIMA模型和不同的gamma/tau组合
for gamma, tau in param_combinations:
    print(f"\n测试方法1: ARIMA模型 (gamma={gamma}, tau={tau})")
    
    # 计算c_sim
    df_train['gamma_test'] = gamma
    df_train['tau_test'] = tau
    df_train['c_sim'] = df_train['LHS'] - (gamma * df_train['flow_launch'])
    
    # 基于c_sim进行ARIMA预测
    predictions = pd.DataFrame()
    
    for country in df_train['country'].unique():
        country_data = df_train[df_train['country'] == country]
        
        try:
            c_series = country_data['c_sim']
            
            if len(c_series) > 5:
                order = find_best_arima_order(c_series)
            else:
                order = (1,0,0)
            
            model = ARIMA(c_series, order=order)
            results = model.fit()
            c_forecast = results.forecast(steps=1)  # 预测2024
            c_forecast = np.maximum(c_forecast, 1e-10)  # 防止负值
            
            country_predictions = pd.DataFrame({
                'year': [2024],
                'country': country,
                'c_pred': c_forecast,
                'arima_order': str(order),
                'method': f'ARIMA',
                'gamma': gamma,
                'tau': tau,
                'param_combo': f'g{gamma}_t{tau}'
            })
            
            last_lhs = country_data['LHS'].iloc[-1]
            country_predictions['flow_launch_pred'] = np.maximum(
                (last_lhs - country_predictions['c_pred']) / gamma, 
                0
            )
            
            predictions = pd.concat([predictions, country_predictions])
            
        except Exception as e:
            print(f"预测{country}时出错: {str(e)}")
            continue
    
    # 计算均衡发射量
    countries = predictions['country'].unique()
    
    # 计算alpha和lambda值
    alphas = {}
    lambdas = {}
    
    for country in countries:
        alphas[country] = 1 / (tau + gamma)
    
    alpha_sum = sum(alphas.values())
    
    for country in countries:
        lambdas[country] = alphas[country] / ((1/tau) + alpha_sum)
    
    # 计算发射量
    predictions['theoretical_flow_launch'] = 0.0
    predictions['sum_lambda_c'] = 0.0
    
    # 获取2023年的total_launch作为初始St-1
    prev_total_launch = df_train[df_train['year'] == 2023]['flow_launch'].sum()
    
    for year in sorted(predictions['year'].unique()):
        year_data = predictions[predictions['year'] == year]
        
        # 计算当年的Σλj(1-cj)
        sum_lambda_c = 0
        for _, row in year_data.iterrows():
            country = row['country']
            c_j = row['c_pred']
            term = lambdas[country] * (1 - c_j)
            sum_lambda_c += term
        
        predictions.loc[predictions['year'] == year, 'sum_lambda_c'] = sum_lambda_c
        
        total_launches = 0
        
        for country in year_data['country'].unique():
            country_mask = (predictions['year'] == year) & (predictions['country'] == country)
            c_i = predictions.loc[country_mask, 'c_pred'].iloc[0]
            
            x_i = alphas[country] * ((1 - c_i) - sum_lambda_c - tau * prev_total_launch)
            x_i = max(x_i, 0)
            
            predictions.loc[country_mask, 'theoretical_flow_launch'] = x_i
            total_launches += x_i
        
        prev_total_launch += total_launches
        predictions.loc[predictions['year'] == year, 'total_launch_t_minus_1'] = prev_total_launch
    
    # 将结果添加到总预测结果中
    all_predictions = pd.concat([all_predictions, predictions])

# 方法2：使用c_sim增长率的线性模型
for gamma, tau in param_combinations:
    print(f"\n测试方法2: c_sim增长率线性模型 (gamma={gamma}, tau={tau})")
    
    # 计算c_sim
    df_train['gamma_test'] = gamma
    df_train['tau_test'] = tau
    df_train['c_sim'] = df_train['LHS'] - (gamma * df_train['flow_launch'])
    
    # 使用c_sim增长率进行预测
    growth_predictions = pd.DataFrame()
    
    for country in df_train['country'].unique():
        country_data = df_train[df_train['country'] == country].sort_values('year')
        
        try:
            # 计算c_sim的年增长率
            country_data['c_sim_growth'] = country_data['c_sim'].pct_change()
            
            # 去除NaN值
            growth_data = country_data.dropna(subset=['c_sim_growth'])
            
            if len(growth_data) > 1:  # 至少需要两个增长率点进行线性回归
                # 准备线性回归数据
                X = growth_data['year'].values.reshape(-1, 1)
                y = growth_data['c_sim_growth'].values
                
                model = LinearRegression()
                model.fit(X, y)
                
                # 预测2024年的增长率
                growth_forecast = model.predict(np.array([[2024]]))
                
                # 获取最后一年的c_sim值
                last_c_sim = country_data['c_sim'].iloc[-1]
                last_year = country_data['year'].iloc[-1]
                
                # 使用预测的增长率计算2024年的c_sim
                c_forecast = last_c_sim * (1 + growth_forecast[0])
                c_forecast = np.maximum(c_forecast, 1e-10)  # 防止负值
                
                country_predictions = pd.DataFrame({
                    'year': [2024],
                    'country': country,
                    'c_pred': c_forecast,
                    'method': 'Growth_Rate',
                    'gamma': gamma,
                    'tau': tau,
                    'param_combo': f'g{gamma}_t{tau}'
                })
                
                last_lhs = country_data['LHS'].iloc[-1]
                country_predictions['flow_launch_pred'] = np.maximum(
                    (last_lhs - country_predictions['c_pred']) / gamma, 
                    0
                )
                
                growth_predictions = pd.concat([growth_predictions, country_predictions])
                
            else:
                # 如果数据不足，使用最后一年的c_sim值
                last_c_sim = country_data['c_sim'].iloc[-1]
                
                country_predictions = pd.DataFrame({
                    'year': [2024],
                    'country': country,
                    'c_pred': last_c_sim,
                    'method': 'Growth_Rate',
                    'gamma': gamma,
                    'tau': tau,
                    'param_combo': f'g{gamma}_t{tau}'
                })
                
                last_lhs = country_data['LHS'].iloc[-1]
                country_predictions['flow_launch_pred'] = np.maximum(
                    (last_lhs - country_predictions['c_pred']) / gamma, 
                    0
                )
                
                growth_predictions = pd.concat([growth_predictions, country_predictions])
                
        except Exception as e:
            print(f"预测{country}时出错: {str(e)}")
            continue
    
    # 计算均衡发射量（与ARIMA方法相同的逻辑）
    countries = growth_predictions['country'].unique()
    
    # 计算alpha和lambda值
    alphas = {}
    lambdas = {}
    
    for country in countries:
        alphas[country] = 1 / (tau + gamma)
    
    alpha_sum = sum(alphas.values())
    
    for country in countries:
        lambdas[country] = alphas[country] / ((1/tau) + alpha_sum)
    
    # 计算发射量
    growth_predictions['theoretical_flow_launch'] = 0.0
    growth_predictions['sum_lambda_c'] = 0.0
    
    # 获取2023年的total_launch作为初始St-1
    prev_total_launch = df_train[df_train['year'] == 2023]['flow_launch'].sum()
    
    for year in sorted(growth_predictions['year'].unique()):
        year_data = growth_predictions[growth_predictions['year'] == year]
        
        # 计算当年的Σλj(1-cj)
        sum_lambda_c = 0
        for _, row in year_data.iterrows():
            country = row['country']
            c_j = row['c_pred']
            term = lambdas[country] * (1 - c_j)
            sum_lambda_c += term
        
        growth_predictions.loc[growth_predictions['year'] == year, 'sum_lambda_c'] = sum_lambda_c
        
        total_launches = 0
        
        for country in year_data['country'].unique():
            country_mask = (growth_predictions['year'] == year) & (growth_predictions['country'] == country)
            c_i = growth_predictions.loc[country_mask, 'c_pred'].iloc[0]
            
            x_i = alphas[country] * ((1 - c_i) - sum_lambda_c - tau * prev_total_launch)
            x_i = max(x_i, 0)
            
            growth_predictions.loc[country_mask, 'theoretical_flow_launch'] = x_i
            total_launches += x_i
        
        prev_total_launch += total_launches
        growth_predictions.loc[growth_predictions['year'] == year, 'total_launch_t_minus_1'] = prev_total_launch
    
    # 将结果添加到总预测结果中
    all_predictions = pd.concat([all_predictions, growth_predictions])

# 方法3：使用c_sim的线性回归模型
for gamma, tau in param_combinations:
    print(f"\n测试方法3: c_sim线性回归模型 (gamma={gamma}, tau={tau})")
    
    # 计算c_sim
    df_train['gamma_test'] = gamma
    df_train['tau_test'] = tau
    df_train['c_sim'] = df_train['LHS'] - (gamma * df_train['flow_launch'])
    
    # 使用线性回归预测c_sim
    linear_predictions = pd.DataFrame()
    
    for country in df_train['country'].unique():
        country_data = df_train[df_train['country'] == country]
        
        try:
            # 准备线性回归数据
            X = country_data['year'].values.reshape(-1, 1)
            y = country_data['c_sim'].values
            
            if len(y) > 1:  # 至少需要两个点进行线性回归
                model = LinearRegression()
                model.fit(X, y)
                
                # 预测2024年
                c_forecast = model.predict(np.array([[2024]]))
                c_forecast = np.maximum(c_forecast, 1e-10)  # 防止负值
                
                country_predictions = pd.DataFrame({
                    'year': [2024],
                    'country': country,
                    'c_pred': c_forecast[0],
                    'method': 'Linear_Regression',
                    'gamma': gamma,
                    'tau': tau,
                    'param_combo': f'g{gamma}_t{tau}'
                })
                
                last_lhs = country_data['LHS'].iloc[-1]
                country_predictions['flow_launch_pred'] = np.maximum(
                    (last_lhs - country_predictions['c_pred']) / gamma, 
                    0
                )
                
                linear_predictions = pd.concat([linear_predictions, country_predictions])
                
            else:
                print(f"{country}数据不足，无法进行线性回归")
                
        except Exception as e:
            print(f"预测{country}时出错: {str(e)}")
            continue
    
    # 计算均衡发射量（与ARIMA方法相同的逻辑）
    countries = linear_predictions['country'].unique()
    
    # 计算alpha和lambda值
    alphas = {}
    lambdas = {}
    
    for country in countries:
        alphas[country] = 1 / (tau + gamma)
    
    alpha_sum = sum(alphas.values())
    
    for country in countries:
        lambdas[country] = alphas[country] / ((1/tau) + alpha_sum)
    
    # 计算发射量
    linear_predictions['theoretical_flow_launch'] = 0.0
    linear_predictions['sum_lambda_c'] = 0.0
    
    # 获取2023年的total_launch作为初始St-1
    prev_total_launch = df_train[df_train['year'] == 2023]['flow_launch'].sum()
    
    for year in sorted(linear_predictions['year'].unique()):
        year_data = linear_predictions[linear_predictions['year'] == year]
        
        # 计算当年的Σλj(1-cj)
        sum_lambda_c = 0
        for _, row in year_data.iterrows():
            country = row['country']
            c_j = row['c_pred']
            term = lambdas[country] * (1 - c_j)
            sum_lambda_c += term
        
        linear_predictions.loc[linear_predictions['year'] == year, 'sum_lambda_c'] = sum_lambda_c
        
        total_launches = 0
        
        for country in year_data['country'].unique():
            country_mask = (linear_predictions['year'] == year) & (linear_predictions['country'] == country)
            c_i = linear_predictions.loc[country_mask, 'c_pred'].iloc[0]
            
            x_i = alphas[country] * ((1 - c_i) - sum_lambda_c - tau * prev_total_launch)
            x_i = max(x_i, 0)
            
            linear_predictions.loc[country_mask, 'theoretical_flow_launch'] = x_i
            total_launches += x_i
        
        prev_total_launch += total_launches
        linear_predictions.loc[linear_predictions['year'] == year, 'total_launch_t_minus_1'] = prev_total_launch
    
    # 将线性回归结果添加到总预测结果中
    all_predictions = pd.concat([all_predictions, linear_predictions])

# 保存所有预测结果
all_predictions.to_csv(f"{results_dir}/all_predictions.csv", index=False)

# 对比2024年预测与实际值
predictions_2024 = all_predictions[all_predictions['year'] == 2024].copy()

# 合并实际2024年数据
actual_2024 = df_2024[['country', 'flow_launch']].rename(columns={'flow_launch': 'actual_flow_launch'})
comparison = pd.merge(predictions_2024, actual_2024, on='country', how='left')

# 计算差异
comparison['absolute_diff'] = abs(comparison['theoretical_flow_launch'] - comparison['actual_flow_launch'])
# 使用自定义函数计算百分比差异，避免除以零的问题
comparison['percentage_diff'] = 100 * comparison['absolute_diff'] / comparison['actual_flow_launch'].clip(lower=0.1)

# 保存比较结果
comparison.to_csv(f"{results_dir}/prediction_vs_actual_2024.csv", index=False)

# 计算每种方法和参数组合的总发射量
total_launches = comparison.groupby(['method', 'gamma', 'tau', 'param_combo']).agg({
    'theoretical_flow_launch': 'sum',
    'actual_flow_launch': 'sum',
    'absolute_diff': 'mean',
    'percentage_diff': 'mean'
}).reset_index()

# 计算总体实际发射量
actual_total = df_2024['flow_launch'].sum()
total_launches['actual_total'] = actual_total

# 保存总发射量结果
total_launches.to_csv(f"{results_dir}/total_launches_comparison.csv", index=False)

# 重点关注中国、美国和CIS的结果
focus_countries = ['China', 'United States', 'Commonwealth of Independent States']
focus_results = comparison[comparison['country'].isin(focus_countries)].copy()
focus_results = focus_results.sort_values(['country', 'method', 'gamma', 'tau'])
focus_results.to_csv(f"{results_dir}/focus_countries_comparison.csv", index=False)

# 打印重点国家的结果
print("\n重点国家预测结果对比:")
for country in focus_countries:
    print(f"\n{country}的预测结果:")
    country_data = focus_results[focus_results['country'] == country]
    for _, row in country_data.iterrows():
        print(f"方法: {row['method']}, gamma: {row['gamma']}, tau: {row['tau']}")
        print(f"  预测发射量: {row['theoretical_flow_launch']:.2f}")
        print(f"  实际发射量: {row['actual_flow_launch']:.2f}")
        print(f"  绝对差异: {row['absolute_diff']:.2f}")
        print(f"  百分比差异: {row['percentage_diff']:.2f}%")

# 创建可视化图表
# 1. 按照参数组合绘制总发射量对比图
methods = total_launches['method'].unique()

# 为每个方法创建一个图表，x轴为参数组合，y轴为发射量
for method in methods:
    method_data = total_launches[total_launches['method'] == method].sort_values(['gamma', 'tau'])
    
    plt.figure(figsize=(15, 8))
    
    # 绘制预测值
    plt.plot(method_data['param_combo'], method_data['theoretical_flow_launch'], 'o-', label='预测总发射量')
    
    # 绘制实际值（水平线）
    plt.axhline(y=actual_total, color='r', linestyle='-', label='实际总发射量')
    
    plt.xlabel('参数组合 (gamma_tau)')
    plt.ylabel('发射量')
    plt.title(f'{method} 方法下不同参数组合的2024年总发射量预测')
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{results_dir}/{method}_total_prediction.png", dpi=300)
    plt.close()

# 2. 为重点国家创建参数敏感性分析图
for country in focus_countries:
    for method in methods:
        country_method_data = focus_results[(focus_results['country'] == country) & 
                                           (focus_results['method'] == method)].sort_values(['gamma', 'tau'])
        
        if not country_method_data.empty:
            plt.figure(figsize=(15, 8))
            
            # 绘制预测值
            plt.plot(country_method_data['param_combo'], country_method_data['theoretical_flow_launch'], 'o-', label='预测发射量')
            
            # 绘制实际值（水平线）
            actual_value = country_method_data['actual_flow_launch'].iloc[0]
            plt.axhline(y=actual_value, color='r', linestyle='-', label='实际发射量')
            
            plt.xlabel('参数组合 (gamma_tau)')
            plt.ylabel('发射量')
            plt.title(f'{country} - {method} 方法下不同参数组合的2024年发射量预测')
            plt.xticks(rotation=45)
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(f"{results_dir}/{country}_{method}_prediction.png", dpi=300)
            plt.close()

# 3. 创建gamma和tau的热力图，展示预测准确性
for method in methods:
    method_data = total_launches[total_launches['method'] == method].copy()
    
    # 创建gamma和tau的网格
    gamma_unique = sorted(method_data['gamma'].unique())
    tau_unique = sorted(method_data['tau'].unique())
    
    # 创建热力图数据
    heatmap_data = np.zeros((len(gamma_unique), len(tau_unique)))
    
    for i, gamma in enumerate(gamma_unique):
        for j, tau in enumerate(tau_unique):
            mask = (method_data['gamma'] == gamma) & (method_data['tau'] == tau)
            if mask.any():
                # 使用绝对差异作为热力图值
                heatmap_data[i, j] = method_data.loc[mask, 'absolute_diff'].values[0]
    
    plt.figure(figsize=(10, 8))
    plt.imshow(heatmap_data, cmap='viridis_r')  # viridis_r使得较低的误差显示为较亮的颜色
    
    # 添加坐标轴标签
    plt.xticks(np.arange(len(tau_unique)), [f"{t:.1e}" for t in tau_unique], rotation=45)
    plt.yticks(np.arange(len(gamma_unique)), [f"{g:.4f}" for g in gamma_unique])
    
    plt.xlabel('tau值')
    plt.ylabel('gamma值')
    plt.title(f'{method} 方法下不同参数组合的预测误差热力图')
    
    plt.colorbar(label='平均绝对误差')
    plt.tight_layout()
    plt.savefig(f"{results_dir}/{method}_error_heatmap.png", dpi=300)
    plt.close()

# 4. 创建一个综合图表，比较三种方法在最佳参数组合下的表现
best_params = total_launches.groupby('method').apply(
    lambda x: x.loc[x['absolute_diff'].idxmin()]
).reset_index(drop=True)

plt.figure(figsize=(12, 8))

x = np.arange(len(methods))
width = 0.35

plt.bar(x - width/2, best_params['theoretical_flow_launch'], width, label='最佳参数下的预测值')
plt.bar(x + width/2, best_params['actual_total'], width, label='实际值')

plt.xlabel('预测方法')
plt.ylabel('总发射量')
plt.title('各方法在最佳参数组合下的2024年总发射量预测')
plt.xticks(x, methods)
plt.legend()
plt.grid(True, axis='y')
plt.tight_layout()
plt.savefig(f"{results_dir}/best_methods_comparison.png", dpi=300)
plt.close()

print(f"\n所有结果已保存到 {results_dir} 目录")

In [ ]:
import pandas as pd
import os
import glob

def combine_csv_data():
    # Get list of all CSV files matching the pattern
    csv_files = glob.glob('consistent_gamma_*_predictions.csv')
    
    # Sort the files to maintain order
    csv_files.sort()
    
    # Initialize an empty DataFrame to store the combined data
    combined_data = pd.DataFrame()
    
    for file in csv_files:
        # Extract gamma value from filename
        gamma_value = file.split('_')[2]
        
        # Read the CSV file
        df = pd.read_csv(file)
        
        # Extract required columns and rename them to include gamma value
        columns_to_extract = {
            'c_pred': f'c_pred_{gamma_value}',
            'theoretical_flow_launch': f'theoretical_flow_launch_{gamma_value}',
     
        }
        
        # If this is the first file, keep year and country columns for reference
        if combined_data.empty:
            extracted_df = df[['year', 'country'] + list(columns_to_extract.keys())].copy()
            extracted_df.rename(columns=columns_to_extract, inplace=True)
            combined_data = extracted_df
        else:
            # For subsequent files, just extract the required columns
            temp_df = df[list(columns_to_extract.keys())].copy()
            temp_df.rename(columns=columns_to_extract, inplace=True)
            
            # Add these columns to combined_data
            combined_data = pd.concat([combined_data, temp_df], axis=1)
    
    # Save the combined data to a new CSV file
    output_file = 'combined_gamma_predictions.csv'
    combined_data.to_csv(output_file, index=False)
    print(f"Combined data saved to {output_file}")
    return combined_data

if __name__ == "__main__":
    combine_csv_data()

In [ ]:
# 反向计算一致的gamma值
def find_consistent_gamma_for_target(df, target_country='United States', target_year=2025, 
                                   target_launch=2500, tau=0.0000062):
    """
    反向计算一个一致的gamma值，使目标国家在目标年份达到指定发射量
    使用同一个gamma值计算c_sim，预测c_pred，并计算发射量
    """
    # 获取所有国家列表
    all_countries = df['country'].unique()
    n = len(all_countries)
    
    # 获取最后一年
    last_year = df['year'].max()
    forecast_steps = target_year - last_year
    
    # 初始累积发射量
    prev_total_launch = 18004.742
    
    print(f"\n反向计算一致的gamma值使{target_country}在{target_year}年达到{target_launch}发射量")
    print(f"国家数量: {n}")
    print(f"当前年份: {last_year}, 预测步数: {forecast_steps}")
    print(f"初始累积发射量: {prev_total_launch}")
    
    # 尝试不同的gamma值
    gamma_values = np.logspace(-6, -2, 50)  # 50个点
    results = []
    
    for gamma in gamma_values:
        try:
            print(f"测试 gamma = {gamma:.6f}")
            
            # 1. 使用当前gamma计算所有国家的c_sim
            df_temp = df.copy()
            df_temp['c_sim'] = df_temp['LHS'] - (gamma * df_temp['flow_launch'])
            
            # 2. 为所有国家预测c_pred
            predictions = pd.DataFrame(columns=['country', 'c_pred'])
            
            for country in all_countries:
                country_data = df_temp[df_temp['country'] == country]
                
                if len(country_data) < 5:  # 数据不足
                    last_c = country_data['c_sim'].iloc[-1] if len(country_data) > 0 else 0.5
                    predictions = predictions.append({'country': country, 'c_pred': last_c}, ignore_index=True)
                    continue
                
                # 使用ARIMA(1,1,1)模型
                try:
                    series = country_data['c_sim']
                    model = ARIMA(series, order=(1,1,1))
                    results_arima = model.fit()
                    forecast = results_arima.forecast(steps=forecast_steps)
                    c_pred = forecast[forecast_steps-1]
                    predictions = predictions.append({'country': country, 'c_pred': c_pred}, ignore_index=True)
                except:
                    # 如果ARIMA模型失败，使用最后的c值
                    last_c = country_data['c_sim'].iloc[-1]
                    predictions = predictions.append({'country': country, 'c_pred': last_c}, ignore_index=True)
            
            # 3. 计算2025年的sum_lambda_c
            alpha = 1 / (tau + gamma)
            lambda_val = alpha / ((1/tau) + (n * alpha))
            
            sum_lambda_c = 0
            for _, row in predictions.iterrows():
                c_j = row['c_pred']
                term = lambda_val * (1 - c_j)
                sum_lambda_c += term
            
            # 4. 计算目标国家的发射量
            target_c = predictions[predictions['country'] == target_country]['c_pred'].values[0]
            
            x_target = alpha * ((1 - target_c) - sum_lambda_c - tau * prev_total_launch)
            x_target = max(x_target, 0)
            
            # 5. 记录结果
            diff = abs(x_target - target_launch)
            results.append({
                'gamma': gamma,
                'c_target': target_c,
                'sum_lambda_c': sum_lambda_c,
                'x_target': x_target,
                'diff': diff
            })
            
            print(f"  c_target: {target_c:.4f}, x_target: {x_target:.2f}, diff: {diff:.2f}")
            
        except Exception as e:
            print(f"  gamma={gamma}时出错: {str(e)}")
            continue
    
    # 找出最佳结果
    if not results:
        print("未找到有效结果")
        return 0.0001
    
    # 转换为DataFrame并按差距排序
    results_df = pd.DataFrame(results)
    best_result = results_df.loc[results_df['diff'].idxmin()]
    
    best_gamma = best_result['gamma']
    best_c = best_result['c_target']
    best_x = best_result['x_target']
    best_diff = best_result['diff']
    
    print(f"\n最佳gamma值: {best_gamma}")
    print(f"{target_country}的c值: {best_c}")
    print(f"使用此gamma计算的发射量: {best_x}")
    print(f"与目标值的差距: {best_diff}")
    
    return best_gamma

# 调用函数
best_gamma = find_consistent_gamma_for_target(df, target_country='United States', 
                                            target_year=2025, target_launch=2500)